# <font color="#418FDE" size="6.5" uppercase>**OLS Ridge**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Fit and interpret ordinary least-squares linear regression models. 
- Apply Ridge regularization for regression and classification tasks. 
- Select Ridge regularization strength using cross-validation. 


## **1. Least Squares Basics**

### **1.1. Assumptions Intuition**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_01_01.jpg?v=1783957989" width="250">



>* OLS summarizes trends with a straight line
>* Nonlinear patterns can make interpretations misleading

>* Residuals should look like random noise
>* Errors need independence and equal spread

>* Check residuals for normal, balanced noise
>* Match assumption strictness to analysis goals



### **1.2. Using LinearRegression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_01_02.jpg?v=1783957991" width="250">



>* Fit straight-line relationships by minimizing squared errors
>* Interpret coefficients as feature effect estimates

>* Prepare numeric outcomes and encoded features
>* Interpret intercepts and unit-based coefficients

>* Interpret coefficients as associations, not causes
>* Evaluate predictions and communicate limitations clearly



In [ ]:
#@title Python Code - Using LinearRegression

# Demonstrate ordinary least squares regression.
# Use NumPy instead of scikit-learn.
# Interpret coefficients with simple housing data.

import numpy as np
import matplotlib.pyplot as plt

# Create tiny housing feature matrix.
square_feet = np.array([800, 1000, 1200, 1500, 1800, 2200])
bedrooms = np.array([1, 2, 2, 3, 3, 4])
prices = np.array([180, 220, 250, 310, 360, 430])

# Validate matching observation counts.
assert square_feet.size == bedrooms.size == prices.size

# Build design matrix with intercept.
ones = np.ones_like(square_feet, dtype=float)
X = np.column_stack([ones, square_feet, bedrooms])
y = prices.astype(float)

# Solve ordinary least squares directly.
coefficients = np.linalg.lstsq(X, y, rcond=None)[0]
intercept, sqft_coef, bedroom_coef = coefficients

# Make predictions for training homes.
predictions = X @ coefficients
errors = y - predictions
rmse = np.sqrt(np.mean(errors ** 2))

# Predict one new example home.
new_home = np.array([1.0, 1600.0, 3.0])
new_prediction = float(new_home @ coefficients)

# Print compact interpretation results.
print(f"Intercept: {intercept:.2f} thousand dollars")
print(f"Square-foot coefficient: {sqft_coef:.3f} thousand per square foot")
print(f"Bedroom coefficient: {bedroom_coef:.2f} thousand per bedroom")
print(f"Training RMSE: {rmse:.2f} thousand dollars")
print(f"Predicted price for 1600 sqft, 3 bedrooms: {new_prediction:.1f}k")

# Plot actual and fitted prices.
plt.figure(figsize=(6, 4))
plt.scatter(square_feet, y, label="Actual prices")
plt.plot(square_feet, predictions, label="OLS fitted values")
plt.xlabel("Home size in square feet")
plt.ylabel("Price in thousand dollars")
plt.title("LinearRegression idea using least squares")
plt.legend()
plt.show()



### **1.3. Residual Diagnostics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_01_03.jpg?v=1783957993" width="250">



>* Residuals reveal model prediction errors
>* Patterns suggest missing or flawed model structure

>* Check residual spread stays roughly constant
>* Look for curved patterns showing nonlinearity

>* Spot outliers and influential data points
>* Judge model trustworthiness before interpreting results



In [ ]:
#@title Python Code - Residual Diagnostics

# Residual diagnostics reveal model problems.
# We simulate apartments with nonlinear rent.
# Plots compare residuals against predictions.

# Import numerical and plotting tools.
import numpy as np
import matplotlib.pyplot as plt

# Make randomness reproducible for learners.
rng = np.random.default_rng(7)

# Create apartment sizes in square feet.
n = 80
size_sqft = rng.uniform(400, 1600, n)

# Create bedrooms related to apartment size.
bedrooms = np.clip((size_sqft / 500).round(), 1, 4)

# Add nonlinear rent pattern and noise.
noise = rng.normal(0, 120, n)
rent = 700 + 1.4 * size_sqft + 90 * bedrooms
rent += 0.0009 * (size_sqft - 1000) ** 2 + noise

# Build a simple OLS design matrix.
X = np.column_stack([np.ones(n), size_sqft, bedrooms])
y = rent.reshape(-1, 1)

# Validate dimensions before fitting.
assert X.shape[0] == y.shape[0]
assert X.shape[1] == 3

# Fit ordinary least squares manually.
beta = np.linalg.lstsq(X, y, rcond=None)[0]
predicted = (X @ beta).ravel()
residuals = rent - predicted

# Summarize residual behavior briefly.
print(f"OLS intercept: ${beta[0,0]:.2f}")
print(f"Size coefficient: ${beta[1,0]:.2f} per square foot")
print(f"Bedroom coefficient: ${beta[2,0]:.2f} per bedroom")
print(f"Mean residual: ${residuals.mean():.2f}")
print(f"Residual standard deviation: ${residuals.std(ddof=3):.2f}")

# Draw one diagnostic residual plot.
plt.figure(figsize=(7, 4))
plt.scatter(predicted, residuals, alpha=0.75)
plt.axhline(0, color="black", linewidth=1)
plt.xlabel("Predicted rent in dollars")
plt.ylabel("Residual in dollars")
plt.title("Residual Diagnostics for OLS Rent Model")
plt.tight_layout()
plt.show()



## **2. Ridge Models**

### **2.1. Ridge Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_02_01.jpg?v=1783957977" width="250">



>* Penalizes large coefficients for stability
>* Improves generalization with related predictors

>* Balances training fit with model simplicity
>* Shrinks coefficients for steadier predictions

>* Shrunken coefficients show direction, not exact effects
>* Ridge improves stability with many related predictors



In [ ]:
#@title Python Code - Ridge Regression

# Ridge regression shrinks unstable linear coefficients.
# This example uses only NumPy operations.
# We compare ordinary least squares and ridge.

import numpy as np
import matplotlib.pyplot as plt

# Set a seed for reproducible synthetic data.
rng = np.random.default_rng(42)

# Create correlated house size features.
square_feet = rng.normal(1800, 350, 40)
bedrooms = square_feet / 650 + rng.normal(0, 0.35, 40)

# Stack features into a small design matrix.
X = np.column_stack([square_feet, bedrooms])
y = 50000 + 145 * square_feet + 12000 * bedrooms

y = y + rng.normal(0, 25000, 40)
assert X.shape == (40, 2)

# Standardize predictors before applying ridge penalties.
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)

X_scaled = (X - X_mean) / X_std
X_design = np.column_stack([np.ones(len(X_scaled)), X_scaled])

# Solve ordinary least squares with linear algebra.
ols_coef = np.linalg.solve(X_design.T @ X_design, X_design.T @ y)

# Penalize feature coefficients, but not the intercept.
alpha = 25.0
penalty = np.diag([0.0, alpha, alpha])

ridge_coef = np.linalg.solve(
    X_design.T @ X_design + penalty,
    X_design.T @ y,
)

# Compare coefficient sizes and training errors.
ols_pred = X_design @ ols_coef
ridge_pred = X_design @ ridge_coef

ols_rmse = np.sqrt(np.mean((y - ols_pred) ** 2))
ridge_rmse = np.sqrt(np.mean((y - ridge_pred) ** 2))

print("NumPy version:", np.__version__)
print("OLS scaled coefficients:", np.round(ols_coef[1:], 1))
print("Ridge scaled coefficients:", np.round(ridge_coef[1:], 1))
print("OLS training RMSE:", round(float(ols_rmse), 2))
print("Ridge training RMSE:", round(float(ridge_rmse), 2))
print("Ridge shrinks coefficients toward zero for stability.")

# Plot actual prices against both model predictions.
plt.figure(figsize=(7, 4))
plt.scatter(y, ols_pred, label="OLS", alpha=0.75)
plt.scatter(y, ridge_pred, label="Ridge", alpha=0.75)

plt.plot([y.min(), y.max()], [y.min(), y.max()], "k--")
plt.xlabel("Actual price in dollars")
plt.ylabel("Predicted price in dollars")
plt.title("OLS versus Ridge Regression")

plt.legend()
plt.tight_layout()
plt.show()



### **2.2. Ridge Classification**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_02_02.jpg?v=1783957987" width="250">



>* Ridge classification predicts categories from features
>* Regularization reduces overfitting and improves generalization

>* Shrinks weights while keeping all predictors
>* Stabilizes classifiers for more reliable predictions

>* Shrunken coefficients need careful interpretation
>* Penalty strength balances flexibility and stability



### **2.3. Scaling Effects**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_02_03.jpg?v=1783957979" width="250">



>* Ridge penalties depend on feature scale
>* Unscaled units can distort coefficient shrinkage

>* Standardize predictors for fair Ridge penalties
>* Scaling improves regression and classification stability

>* Scale within training to tune Ridge fairly
>* Standardized coefficients remain interpretable and comparable



In [ ]:
#@title Python Code - Scaling Effects

# Ridge penalties depend on feature scale.
# This example uses only NumPy.
# Standardizing makes shrinkage more comparable.

import numpy as np
import matplotlib.pyplot as plt

# Create deterministic synthetic housing data.
rng = np.random.default_rng(7)
n_samples = 80
size_feet = rng.normal(1800, 350, n_samples)

# Add a small-scale rating feature.
rating_points = rng.normal(6.5, 1.2, n_samples)
noise = rng.normal(0, 25000, n_samples)
price = 60000 + 120 * size_feet + 18000 * rating_points + noise

# Build a design matrix with intercept.
ones = np.ones(n_samples)
X_raw = np.column_stack([ones, size_feet, rating_points])
y = price

# Define a small Ridge solver.
def ridge_fit(X, y, alpha):
    penalty = np.eye(X.shape[1])
    penalty[0, 0] = 0
    return np.linalg.solve(X.T @ X + alpha * penalty, X.T @ y)

# Standardize predictors, not the intercept.
means = X_raw[:, 1:].mean(axis=0)
scales = X_raw[:, 1:].std(axis=0)
assert np.all(scales > 0)
X_scaled = np.column_stack([ones, (X_raw[:, 1:] - means) / scales])

# Fit Ridge with the same alpha.
alpha_value = 1000.0
coef_raw = ridge_fit(X_raw, y, alpha_value)
coef_scaled = ridge_fit(X_scaled, y, alpha_value)

# Compare coefficient magnitudes.
print(f"Raw size coefficient: {coef_raw[1]:,.2f}")
print(f"Raw rating coefficient: {coef_raw[2]:,.2f}")
print(f"Scaled size coefficient: {coef_scaled[1]:,.2f}")
print(f"Scaled rating coefficient: {coef_scaled[2]:,.2f}")
print("Scaling lets Ridge penalize comparable feature changes.")

# Plot the coefficient comparison.
labels = ["size", "rating"]
raw_values = np.abs(coef_raw[1:])
scaled_values = np.abs(coef_scaled[1:])

# Draw one compact teaching plot.
positions = np.arange(len(labels))
width = 0.35
plt.figure(figsize=(7, 4))
plt.bar(positions - width / 2, raw_values, width, label="raw")

plt.bar(positions + width / 2, scaled_values, width, label="scaled")
plt.xticks(positions, labels)
plt.ylabel("absolute coefficient")
plt.title("Ridge coefficients change after scaling")

plt.legend()
plt.tight_layout()
plt.show()



## **3. Ridge Selection**

### **3.1. Ridge Cross Validation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_03_01.jpg?v=1783957985" width="250">



>* Ridge shrinks coefficients to improve stability
>* Cross validation chooses the best shrinkage strength

>* Split data to test Ridge settings
>* Choose shrinkage that predicts new cases

>* Choose balance, not best training fit
>* Prevent leakage for reliable future predictions



In [ ]:
#@title Python Code - Ridge Cross Validation

# Demonstrate ridge cross validation clearly.
# Use only NumPy and Matplotlib.
# Compare validation errors across alphas.

import numpy as np
import matplotlib.pyplot as plt

# Create reproducible correlated regression data.
rng = np.random.default_rng(7)
n_samples, n_features = 80, 6
base_signal = rng.normal(size=(n_samples, 1))

# Build overlapping predictors with noise.
feature_noise = rng.normal(scale=0.25, size=(n_samples, n_features))
X = base_signal + feature_noise
true_beta = np.array([3.0, -2.0, 1.5, 0.0, 0.0, 0.5])

# Generate a noisy target variable.
y_noise = rng.normal(scale=1.2, size=n_samples)
y = X @ true_beta + y_noise
assert X.shape[0] == y.shape[0]

# Standardize features before ridge fitting.
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_scaled = (X - X_mean) / X_std

# Define candidate regularization strengths.
alphas = np.array([0.0, 0.1, 1.0, 10.0, 100.0])
k_folds = 5
fold_ids = np.arange(n_samples) % k_folds

# Fit ridge using the closed form solution.
def ridge_fit_predict(X_train, y_train, X_valid, alpha):
    identity = np.eye(X_train.shape[1])
    weights = np.linalg.solve(X_train.T @ X_train + alpha * identity, X_train.T @ y_train)
    return X_valid @ weights

# Evaluate each alpha with cross validation.
mean_errors = []
for alpha in alphas:
    fold_errors = []
    for fold in range(k_folds):
        valid_mask = fold_ids == fold
        train_mask = ~valid_mask

        # Predict held out rows only.
        predictions = ridge_fit_predict(
            X_scaled[train_mask], y[train_mask], X_scaled[valid_mask], alpha
        )
        mse = np.mean((y[valid_mask] - predictions) ** 2)
        fold_errors.append(mse)

    # Store average validation error.
    mean_errors.append(float(np.mean(fold_errors)))

# Select the alpha with lowest validation error.
best_index = int(np.argmin(mean_errors))
best_alpha = float(alphas[best_index])
best_error = float(mean_errors[best_index])

# Print a compact teaching summary.
print(f"NumPy version: {np.__version__}")
print("Ridge cross validation results:")
for alpha, error in zip(alphas, mean_errors):
    print(f"alpha={alpha:6.1f}  mean validation MSE={error:6.3f}")
print(f"Selected alpha: {best_alpha:.1f}, validation MSE: {best_error:.3f}")

# Plot validation error against regularization strength.
plt.figure(figsize=(7, 4))
plt.plot(alphas, mean_errors, marker="o")
plt.xscale("symlog", linthresh=0.1)
plt.xlabel("Ridge alpha")
plt.ylabel("Mean validation MSE")
plt.title("Choosing Ridge Regularization by Cross Validation")
plt.grid(True, alpha=0.3)
plt.show()



### **3.2. Multioutput Ridge**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_03_02.jpg?v=1783957981" width="250">



>* Predict multiple related outcomes together
>* Stabilize parallel regressions with Ridge

>* Regularization strength affects every predicted outcome
>* Cross-validation chooses the best overall compromise

>* Shared predictors, separate outcome relationships
>* Cross-validate using balanced output performance



In [ ]:
#@title Python Code - Multioutput Ridge

# Multioutput Ridge predicts several targets together.
# Cross validation chooses stable shrinkage strength.
# This example uses only NumPy.

import numpy as np
import matplotlib.pyplot as plt

# Set a seed for reproducible results.
rng = np.random.default_rng(42)

# Create small correlated feature data.
n_samples, n_features, n_outputs = 90, 6, 3
base = rng.normal(size=(n_samples, 1))

# Add related predictors with noise.
noise = rng.normal(scale=0.35, size=(n_samples, n_features))
X = base @ np.ones((1, n_features)) + noise

# Define separate relationships for each output.
true_coef = np.array([[2.0, -1.0, 0.5], [1.5, 0.0, -0.5], [0.0, 1.2, 1.0], [-1.0, 0.8, 0.0], [0.5, -0.4, 1.5], [1.0, 0.3, -1.2]])
Y = X @ true_coef + rng.normal(scale=0.8, size=(n_samples, n_outputs))

# Validate the multioutput target shape.
assert X.shape == (n_samples, n_features)
assert Y.shape == (n_samples, n_outputs)

# Standardize features and targets for fair fitting.
X_mean, X_std = X.mean(axis=0), X.std(axis=0)
Y_mean, Y_std = Y.mean(axis=0), Y.std(axis=0)

# Avoid division by zero safely.
X_std = np.where(X_std == 0, 1.0, X_std)
Y_std = np.where(Y_std == 0, 1.0, Y_std)

# Build standardized arrays for modeling.
Xs = (X - X_mean) / X_std
Ys = (Y - Y_mean) / Y_std

# Define a compact Ridge solver.
def ridge_fit_predict(Xtr, Ytr, Xte, alpha):
    penalty = alpha * np.eye(Xtr.shape[1])
    coef = np.linalg.solve(Xtr.T @ Xtr + penalty, Xtr.T @ Ytr)
    return Xte @ coef

# Prepare candidate regularization strengths.
alphas = np.array([0.01, 0.1, 1.0, 10.0, 100.0])
fold_ids = np.arange(n_samples) % 5

# Store average validation errors.
cv_errors = []

# Evaluate each alpha across folds.
for alpha in alphas:
    fold_errors = []
    for fold in range(5):
        test_mask = fold_ids == fold
        pred = ridge_fit_predict(Xs[~test_mask], Ys[~test_mask], Xs[test_mask], alpha)
        fold_errors.append(np.mean((Ys[test_mask] - pred) ** 2))
    cv_errors.append(np.mean(fold_errors))

# Select the alpha with lowest error.
cv_errors = np.array(cv_errors)
best_alpha = alphas[np.argmin(cv_errors)]

# Fit final model using selected alpha.
final_pred = ridge_fit_predict(Xs, Ys, Xs, best_alpha)
per_output_mse = np.mean((Ys - final_pred) ** 2, axis=0)

# Print concise teaching results.
print(f"NumPy version: {np.__version__}")
print(f"Feature shape: {X.shape}, target shape: {Y.shape}")
print(f"Best Ridge alpha from CV: {best_alpha}")
print(f"Average CV MSE: {cv_errors.min():.3f}")
print(f"Per-output training MSE: {np.round(per_output_mse, 3)}")

# Plot cross-validation error by alpha.
plt.figure(figsize=(6, 4))
plt.semilogx(alphas, cv_errors, marker="o")
plt.xlabel("Ridge alpha")
plt.ylabel("Mean validation MSE")
plt.title("Multioutput Ridge alpha selection")
plt.grid(True, alpha=0.3)
plt.show()



### **3.3. Interpreting Ridge Coefficients**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_03_03.jpg?v=1783957983" width="250">



>* Ridge coefficients show adjusted predictor effects
>* Shrinkage improves stability with correlated features

>* Regularization strength controls coefficient shrinkage
>* Cross-validation balances interpretation with predictive reliability

>* Standardize features before comparing Ridge coefficients
>* Correlated predictors share predictive influence



In [ ]:
#@title Python Code - Interpreting Ridge Coefficients

# Ridge coefficients need careful interpretation.
# Standardization makes coefficient sizes comparable.
# Cross validation selects useful shrinkage.

import numpy as np
import matplotlib.pyplot as plt

# Create a small reproducible housing dataset.
rng = np.random.default_rng(7)
n_samples = 80
size_sqft = rng.normal(1800, 350, n_samples)

# Build correlated predictors with different scales.
bedrooms = size_sqft / 650 + rng.normal(0, 0.35, n_samples)
bathrooms = size_sqft / 900 + rng.normal(0, 0.25, n_samples)
income_k = rng.normal(85, 18, n_samples)

# Stack predictors into one design matrix.
X = np.column_stack((size_sqft, bedrooms, bathrooms, income_k))
true_beta = np.array([0.12, 18.0, 25.0, 1.4])
y = X @ true_beta + rng.normal(0, 35, n_samples)

# Standardize features before applying Ridge penalties.
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_scaled = (X - X_mean) / X_std

# Add an intercept column after standardization.
ones = np.ones((n_samples, 1))
X_design = np.column_stack((ones, X_scaled))

# Define a compact Ridge solver.
def ridge_fit(Xmat, yvec, alpha):
    penalty = np.eye(Xmat.shape[1])
    penalty[0, 0] = 0.0
    left = Xmat.T @ Xmat + alpha * penalty
    right = Xmat.T @ yvec
    return np.linalg.solve(left, right)

# Compare weak and strong regularization strengths.
alphas = [0.01, 1.0, 30.0]
feature_names = ["size_sqft", "bedrooms", "bathrooms", "income_k"]
coefs = [ridge_fit(X_design, y, a)[1:] for a in alphas]

# Print a short interpretation table.
print("Ridge coefficients use standardized predictors.")
for alpha, coef in zip(alphas, coefs):
    rounded = np.round(coef, 2)
    print(f"alpha={alpha:>5}: {dict(zip(feature_names, rounded))}")

# Plot shrinkage across regularization strengths.
for index, name in enumerate(feature_names):
    values = [coef[index] for coef in coefs]
    plt.plot(alphas, values, marker="o", label=name)

# Label the single teaching plot.
plt.xscale("log")
plt.xlabel("Ridge alpha")
plt.ylabel("Standardized coefficient")
plt.title("Ridge shrinkage changes coefficient interpretation")

# Show how related predictors share influence.
plt.axhline(0, color="black", linewidth=0.8)
plt.legend()
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**OLS Ridge**</font>


In this lecture, you learned to:
- Fit and interpret ordinary least-squares linear regression models. 
- Apply Ridge regularization for regression and classification tasks. 
- Select Ridge regularization strength using cross-validation. 

In the next Lecture (Lecture B), we will go over 'Lasso Elastic Net'